#  Código usando Dask-Awkward

In [5]:
import uproot
import awkward as ak
import dask_awkward as dak
import dask
import numpy as np
from dask.diagnostics import ProgressBar

# ==============================================================================
# FUNÇÃO DE TRUTH MATCHING (Retornando apenas a Máscara)
# ==============================================================================
def get_truth_match_mask(pdg, mother):
    
    local_idx = ak.local_index(pdg)
    safe_mom      = ak.where(mother >= 0, mother, 0)
    mom_pdg       = pdg[safe_mom]
    mom_pdg       = ak.where(mother >= 0, mom_pdg, -999)

    safe_grandmom = safe_mom[safe_mom]
    grandmom_pdg  = pdg[safe_grandmom]
    grandmom_pdg  = ak.where(safe_mom >= 0, grandmom_pdg, -999)

    safe_greatgrandmom = safe_mom[safe_grandmom]
    greatgrandmom_pdg  = pdg[safe_greatgrandmom]
    greatgrandmom_pdg  = ak.where(safe_grandmom >= 0, greatgrandmom_pdg, -999)

   
    B_ancestor_idx = ak.full_like(pdg, -1)
    B_ancestor_idx = ak.where((pdg == 511), local_idx, B_ancestor_idx)
    
    # K*0 e J/psi filhos diretos do B0
    B_ancestor_idx = ak.where((pdg == 313) & (mom_pdg == 511), safe_mom, B_ancestor_idx)
    B_ancestor_idx = ak.where((pdg == 443) & (mom_pdg == 511), safe_mom, B_ancestor_idx)

    # γ filho do J/psi, neto do B0
    B_ancestor_idx = ak.where((pdg == 22) & (mom_pdg == 443) & (grandmom_pdg == 511), safe_grandmom, B_ancestor_idx)
    
    # K+ e π-: filhos do K*0, netos do B0
    is_K  = (pdg == 321)  & (mom_pdg == 313) & (grandmom_pdg == 511)
    is_pi = (pdg == -211) & (mom_pdg == 313) & (grandmom_pdg == 511)

    B_ancestor_idx = ak.where(is_K,  safe_grandmom, B_ancestor_idx)
    B_ancestor_idx = ak.where(is_pi, safe_grandmom, B_ancestor_idx)

    is_mu_m = ((pdg == 13)  & (mom_pdg == 22) & (grandmom_pdg == 443) & (greatgrandmom_pdg == 511))
    is_mu_p = ((pdg == -13) & (mom_pdg == 22) & (grandmom_pdg == 443) & (greatgrandmom_pdg == 511))

    B_ancestor_idx = ak.where(is_mu_m | is_mu_p, safe_greatgrandmom, B_ancestor_idx)

    anc_kstar = B_ancestor_idx[(pdg == 313) & (mom_pdg == 511)]
    anc_jpsi  = B_ancestor_idx[(pdg == 443) & (mom_pdg == 511)]
    anc_K     = B_ancestor_idx[is_K]
    anc_pi    = B_ancestor_idx[is_pi]
    anc_mu_m  = B_ancestor_idx[is_mu_m]
    anc_mu_p  = B_ancestor_idx[is_mu_p]
    p_anc     = B_ancestor_idx[:, :, np.newaxis]

    has_1_kstar = ak.sum(p_anc == anc_kstar[:, np.newaxis, :], axis=-1) == 1
    has_1_jpsi  = ak.sum(p_anc == anc_jpsi[:, np.newaxis, :],  axis=-1) == 1
    has_1_K     = ak.sum(p_anc == anc_K[:, np.newaxis, :],     axis=-1) == 1
    has_1_pi    = ak.sum(p_anc == anc_pi[:, np.newaxis, :],    axis=-1) == 1
    has_1_mu_m  = ak.sum(p_anc == anc_mu_m[:, np.newaxis, :],  axis=-1) == 1
    has_1_mu_p  = ak.sum(p_anc == anc_mu_p[:, np.newaxis, :],  axis=-1) == 1

    is_valid_chain = (has_1_kstar & has_1_jpsi & has_1_K & has_1_pi & has_1_mu_m  & has_1_mu_p)
    particle_mask = is_valid_chain & (B_ancestor_idx >= 0)

    return particle_mask


file_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoJpsiKstar_Jpsito2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoJpsiKstar_NoFilter_2022_retry1/260427_173326/0000/BPH_MC_NANO_1.root"
print("Abrindo arquivo e montando grafo lazy...")

arrays = uproot.dask(
    {file_path: "Events"},
    expressions=[
        "BPHGenPart_pdgId",
        "BPHGenPart_genPartIdxMother",
        "BPHGenPart_mass"
    ],
    library="ak"
)

raw_pdg    = arrays["BPHGenPart_pdgId"]
raw_mother = arrays["BPHGenPart_genPartIdxMother"]
raw_mass   = arrays["BPHGenPart_mass"]

print(f"Número de partições: {raw_pdg.npartitions}")

# 1. Mapeia a função diretamente para criar a máscara lazy
mask_lazy = dak.map_partitions(
    get_truth_match_mask,
    raw_pdg,
    raw_mother
)

# 2. Aplica a máscara nos arrays
pdg_masked_lazy  = raw_pdg[mask_lazy]
mass_masked_lazy = raw_mass[mask_lazy]

# 3. Remove os eventos que ficaram vazios após o mascaramento
event_mask_lazy = dak.num(pdg_masked_lazy) > 0

pdg_final_lazy  = pdg_masked_lazy[event_mask_lazy]
mass_final_lazy = mass_masked_lazy[event_mask_lazy]

print("\nExecutando com barra de progresso...")
with ProgressBar():
    # dask.compute processa as coleções em paralelo e retorna arrays na memória (Awkward Arrays)
    pdg_clean, mass_clean = dask.compute(pdg_final_lazy, mass_final_lazy)

print(f"\nTotal de eventos de SINAL retidos: {len(pdg_clean)}")

for i in range(min(5, len(pdg_clean))):
    print(f"\nEvento {i}:")
    print(f"  PDG  : {pdg_clean[i].tolist()}")
    print(f"  Mass : {mass_clean[i].tolist()}")

Abrindo arquivo e montando grafo lazy...
Número de partições: 1

Executando com barra de progresso...
[########################################] | 100% Completed | 101.70 ms
[########################################] | 100% Completed | 2.37 ss

Total de eventos de SINAL retidos: 23737

Evento 0:
  PDG  : [511, 443, -13, 22, 13, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.96484375, 0.494140625, 0.0]

Evento 1:
  PDG  : [511, 443, -13, 22, 13, 22, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.919921875, 0.494140625, 0.0]

Evento 2:
  PDG  : [511, 443, -13, 22, 13, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.87109375, 0.494140625, 0.0]

Evento 3:
  PDG  : [511, 443, -13, 22, 13, 22, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.880859375, 0.494140625, 0.0]

Evento 4:
  PDG  : [511, 443, -13, 22, 13, 22, 22, 22, 313, 321, -211]
  Mass : [5.28125, 3.09375, 0.0, 0.0, 0.0, 0.0, 0.0, 0

# Código usando Loop Python + Numba

In [8]:
import uproot
import awkward as ak
import numba as nb
import numpy as np

@nb.njit
def build_genealogy_masks(pdgs_events, moms_events):
    n_events = len(pdgs_events)
    event_mask = np.zeros(n_events, dtype=np.bool_)
    particle_masks = nb.typed.List()
    
    for i in range(n_events):
        pdgs = pdgs_events[i]
        moms = moms_events[i]
        n_parts = len(pdgs)
        
        p_mask = np.zeros(n_parts, dtype=np.bool_)
        found_signal = False
        
        for j in range(n_parts):
            if pdgs[j] == 511:
                b0_idx = j
                
                # 1. Procurar Filhas Diretas do B0 e contar (exigir 1 de cada)
                kstar_count, jpsi_count = 0, 0
                kstar_idx, jpsi_idx = -1, -1
                for k in range(n_parts):
                    if moms[k] == b0_idx:
                        if pdgs[k] == 313:
                            kstar_count += 1
                            kstar_idx = k
                        elif pdgs[k] == 443:
                            jpsi_count += 1
                            jpsi_idx = k
                            
                if kstar_count != 1 or jpsi_count != 1:
                    continue
                
                # 2. Procurar Kaon e Pion (exigir 1 de cada)
                kaon_count, pion_count = 0, 0
                kaon_idx, pion_idx = -1, -1
                for k in range(n_parts):
                    if moms[k] == kstar_idx:
                        if pdgs[k] == 321: 
                            kaon_count += 1
                            kaon_idx = k
                        elif pdgs[k] == -211: 
                            pion_count += 1
                            pion_idx = k
                            
                if kaon_count != 1 or pion_count != 1:
                    continue

                # 3. Procurar TODOS os Fótons filhos do J/psi
                photon_indices = nb.typed.List.empty_list(nb.int64)
                for k in range(n_parts):
                    if moms[k] == jpsi_idx and pdgs[k] == 22:
                        photon_indices.append(k)
                        
                if len(photon_indices) == 0:
                    continue

                # 4. Procurar Múons cujas mães sejam esses Fótons
                mu_plus_count, mu_minus_count = 0, 0
                mu_plus_idx, mu_minus_idx = -1, -1
                
                for k in range(n_parts):
                    if pdgs[k] == -13: # mu+
                        if moms[k] in photon_indices:
                            mu_plus_count += 1
                            mu_plus_idx = k
                    elif pdgs[k] == 13: # mu-
                        if moms[k] in photon_indices:
                            mu_minus_count += 1
                            mu_minus_idx = k
                            
                # 5. VEREDITO: Exige EXATAMENTE 1 mu+ e 1 mu- na cadeia
                if mu_plus_count == 1 and mu_minus_count == 1:
                    found_signal = True
                    
                    p_mask[b0_idx] = True
                    p_mask[kstar_idx] = True
                    p_mask[jpsi_idx] = True
                    p_mask[kaon_idx] = True
                    p_mask[pion_idx] = True
                    p_mask[mu_plus_idx] = True
                    p_mask[mu_minus_idx] = True
                    
                    # Salva todos os fótons originados do J/psi (igual ao Awkward)
                    for p in photon_indices:
                        p_mask[p] = True
                    
        event_mask[i] = found_signal
        particle_masks.append(p_mask)
        
    return event_mask, particle_masks

file_path = "/cms/store/user/tdeandra/BPH_NanoAOD_MC/BdtoJpsiKstar_Jpsito2Mu_KstartoKPi_TuneCP5_13p6TeV_pythia8-evtgen/BPH_Nano_MC_BdtoJpsiKstar_NoFilter_2022_retry1/260427_173326/0000/BPH_MC_NANO_1.root"

print(f"Abrindo arquivo e carregando arrays...")
file = uproot.open(file_path)
tree = file["Events"]
pdgs = tree["BPHGenPart_pdgId"].array()
moms = tree["BPHGenPart_genPartIdxMother"].array()

print("\nVerificando a genealogia no Numba...")
event_mask, particle_masks_numba = build_genealogy_masks(pdgs, moms)

ak_particle_mask = ak.Array(list(particle_masks_numba))
pdgs_validos = pdgs[event_mask]
mask_valida = ak_particle_mask[event_mask]
pdgs_final = pdgs_validos[mask_valida]

print(f"Total de eventos de SINAL retidos: {len(pdgs_final)}")

Abrindo arquivo e carregando arrays...

Verificando a genealogia no Numba...
Total de eventos de SINAL retidos: 23737


In [9]:
for i in range(min(5, len(pdgs_final))):
    print(f"\nEvento {i}:")
    print(f"  PDG  : {pdgs_final[i].tolist()}")
    print(f"  Mass : {pdgs_final[i].tolist()}")


Evento 0:
  PDG  : [511, 443, -13, 22, 13, 22, 313, 321, -211]
  Mass : [511, 443, -13, 22, 13, 22, 313, 321, -211]

Evento 1:
  PDG  : [511, 443, -13, 22, 13, 22, 22, 313, 321, -211]
  Mass : [511, 443, -13, 22, 13, 22, 22, 313, 321, -211]

Evento 2:
  PDG  : [511, 443, -13, 22, 13, 22, 313, 321, -211]
  Mass : [511, 443, -13, 22, 13, 22, 313, 321, -211]

Evento 3:
  PDG  : [511, 443, -13, 22, 13, 22, 22, 313, 321, -211]
  Mass : [511, 443, -13, 22, 13, 22, 22, 313, 321, -211]

Evento 4:
  PDG  : [511, 443, -13, 22, 13, 22, 22, 22, 313, 321, -211]
  Mass : [511, 443, -13, 22, 13, 22, 22, 22, 313, 321, -211]
